In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path().cwd() / "Data-LI"
LI_ACCOUNT_DATA_PATH = DATA_DIR / "LI-Small_accounts.csv"
LI_TRANSACTIONS_DATA_PATH = DATA_DIR / "LI-Small_Trans.csv"

accounts_df = pd.read_csv(LI_ACCOUNT_DATA_PATH)
trans_df = pd.read_csv(LI_TRANSACTIONS_DATA_PATH)

In [2]:
accounts_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 712688 entries, 0 to 712687
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype
---  ------          --------------   -----
 0   Bank Name       712688 non-null  str  
 1   Bank ID         712688 non-null  int64
 2   Account Number  712688 non-null  str  
 3   Entity ID       712688 non-null  str  
 4   Entity Name     712688 non-null  str  
dtypes: int64(1), str(4)
memory usage: 27.2 MB


In [3]:
trans_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6924049 entries, 0 to 6924048
Data columns (total 11 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Timestamp           str    
 1   From Bank           int64  
 2   Account             str    
 3   To Bank             int64  
 4   Account.1           str    
 5   Amount Received     float64
 6   Receiving Currency  str    
 7   Amount Paid         float64
 8   Payment Currency    str    
 9   Payment Format      str    
 10  Is Laundering       int64  
dtypes: float64(2), int64(3), str(6)
memory usage: 581.1 MB


In [4]:
trans_df['Timestamp'] = pd.to_datetime(trans_df['Timestamp'])
trans_df = trans_df.rename(columns={'Account':'From Account', 'Account.1':'To Account'})
accounts_df = accounts_df.rename(columns={'Account Number': 'Account'})

accounts_df[
    'Universal_Account_ID'
    ] = accounts_df['Bank ID'].astype(str) + "_" + accounts_df['Account'].astype(str)
trans_df[
    'From_Universal_ID'
    ] = trans_df['From Bank'].astype(str) + "_" + trans_df['From Account'].astype(str)
trans_df[
    'To_Universal_ID'
         ] = trans_df['To Bank'].astype(str) + "_" + trans_df['To Account'].astype(str)

In [ ]:

FX_RATES_TO_USD = {
    'US Dollar': 1.0,
    'Euro': 1.00,
    'UK Pound': 1.15,
    'Swiss Franc': 1.03,
    'Canadian Dollar': 0.76,
    'Australian Dollar': 0.68,
    'Yen': 0.0070,            
    'Yuan': 0.144,            
    'Brazil Real': 0.193,     
    'Mexican Peso': 0.050,    
    'Rupee': 0.0125,          
    'Ruble': 0.0165,          
    'Shekel': 0.292,          
    'Saudi Riyal': 0.266,     
    'Bitcoin': 20000.0        
}

trans_df['Payment_FX_Rate'] = trans_df['Payment Currency'].map(FX_RATES_TO_USD).fillna(1.0)
trans_df['Receiving_FX_Rate'] = trans_df['Receiving Currency'].map(FX_RATES_TO_USD).fillna(1.0)

trans_df['Amount_Paid_USD'] = trans_df['Amount Paid'] * trans_df['Payment_FX_Rate']
trans_df['Amount_Received_USD'] = trans_df['Amount Received'] * trans_df['Receiving_FX_Rate']

trans_df = trans_df.drop(columns=['Payment_FX_Rate', 'Receiving_FX_Rate'])

print(trans_df[['Payment Currency', 'Amount Paid', 'Amount_Paid_USD']].head())

In [ ]:
cols_to_clean = [col for col in trans_df.columns if 'Entity Name' in col or 'Universal_Account_ID' in col]
trans_df = trans_df.drop(columns=cols_to_clean, errors='ignore')

trans_df = trans_df.merge(
    accounts_df[['Universal_Account_ID', 'Entity Name']], 
    left_on='From_Universal_ID', 
    right_on='Universal_Account_ID', 
    how='left'
)

trans_df = trans_df_ent.drop(columns=['Universal_Account_ID'])

print(trans_df.info())

In [ ]:
trans_df['Entity_Type'] = trans_df['Entity Name'].str.replace(r'\s*#\d+', '', regex=True)

trans_df = trans_df.drop(columns=['Entity Name'])

print(trans_df['Entity_Type'].value_counts(normalize=True))

In [ ]:
import category_encoders as ce

count_enc = ce.CountEncoder(cols=['Entity_Type'], normalize=True)

trans_df['Entity_Frequency'] = count_enc.fit_transform(trans_df['Entity_Type'])

trans_df = trans_df.drop(columns=['Entity_Type'])

print("Distribuição das Frequências (Contexto Injetado):")
print(trans_df['Entity_Frequency'].value_counts().head())

In [ ]:
trans_df = trans_df.sort_values(by='Timestamp').reset_index(drop=True)

windows = ['1h', '3h', '6h', '12h', '24h']

temp_df = trans_df.set_index('Timestamp')

grouped_from = temp_df.groupby('From_Universal_ID')['Amount Paid']
grouped_to = temp_df.groupby('To_Universal_ID')['Amount Paid']
for window in windows:    
    roll_from = grouped_from.rolling(window).agg(['sum', 'count']).reset_index()
    
    roll_from = roll_from.rename(columns={
        'sum': f'from_vol_{window}',
        'count': f'from_count_{window}'
    })
    
    roll_from = roll_from.drop_duplicates(subset=['From_Universal_ID', 'Timestamp'], keep='last')
    trans_df = trans_df.merge(roll_from, on=['From_Universal_ID', 'Timestamp'], how='left')
    
    roll_to = grouped_to.rolling(window).agg(['sum', 'count']).reset_index()
    
    roll_to = roll_to.rename(columns={
        'sum': f'to_vol_{window}',
        'count': f'to_count_{window}'
    })
    
    roll_to = roll_to.drop_duplicates(subset=['To_Universal_ID', 'Timestamp'], keep='last')
    trans_df = trans_df.merge(roll_to, on=['To_Universal_ID', 'Timestamp'], how='left')